# Thử crawl 97 hostname trong ViBioMIR

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook tự chứa **đúng 97 URL mẫu đã chọn**, mỗi hostname 1 URL từ
`links_corpus.parquet` (4.394.718 dòng; 97 hostname; 65 domain gốc; seed 42).
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler hiện tại trong repo: kiểm tra robots.txt, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Kết quả sẽ khác theo thời điểm và môi trường mạng.

- **VAO_DUOC**: URL cuối trả HTTP 2xx, kể cả khi trích xuất/QC thất bại.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được HTTP 2xx của trang đích.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Dừng do robots là kết quả của chính sách crawler, không chứng minh website
không mở được bằng trình duyệt. Notebook chưa render JavaScript hay OCR.
Một mẫu mỗi hostname chưa đại diện cho toàn bộ URL trên website.


## 1. Cấu hình

Mặc định chạy cả 97 URL. Có thể đặt `MAX_NEW = 10` để thử trước, rồi đổi
thành `None` và chỉ chạy lại ô crawl để làm tiếp. Checkpoint đã có, kể cả
lỗi, được bỏ qua. Muốn thử lại tất cả URL từ đầu, chạy lại ô cấu hình để
tạo thư mục mới. `RESUME_RUN_DIR` dùng để tiếp tục thư mục output đã có.


In [ ]:
WORKERS = 8
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
OUTPUT_BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
               else Path('/content') if Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

97 mẫu giữ nguyên ID/URL nguồn. Ô code có thể mở để kiểm tra danh sách
và snapshot. Crawler được kiểm tra SHA-256 trước khi import.


In [ ]:
SAMPLES = [{'id': 583, 'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai', 'domain': 'special.vietnamplus.vn'},
 {'id': 1143,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-nguoi-lon-cung-dinh-thuy-dau?id=1e8dc2c0-88db-e42e-d5db-cb9137df465f',
  'domain': 'bachmai.gov.vn'},
 {'id': 13933,
  'url': 'https://baoangiang.com.vn/who-canh-bao-nguy-co-lay-lan-cao-cua-virus-chikungunya-a424815.html',
  'domain': 'baoangiang.com.vn'},
 {'id': 23544,
  'url': 'https://baocantho.com.vn/tai-sao-chung-ta-e-oc-moi-khi-cao-mong-tay-len-bang-a72145.html',
  'domain': 'baocantho.com.vn'},
 {'id': 26481,
  'url': 'https://baochinhphu.vn/cac-dia-phuong-can-day-nhanh-tien-do-tiem-chung-vaccine-covid-19-10222060115443633.htm',
  'domain': 'baochinhphu.vn'},
 {'id': 34798,
  'url': 'https://baodanang.vn/khong-chu-quan-voi-benh-glaucoma-3239403.html',
  'domain': 'baodanang.vn'},
 {'id': 44335,
  'url': 'https://baogialai.com.vn/gia-lai-huong-ung-ngay-asean-phong-chong-benh-sot-xuat-huyet-post280882.html',
  'domain': 'baogialai.com.vn'},
 {'id': 58905,
  'url': 'https://baohaiphong.vn/tat-ca-benh-nhan-covid-19-cua-hai-duong-deu-da-xuat-vien-353669.html',
  'domain': 'baohaiphong.vn'},
 {'id': 64079,
  'url': 'https://baolangson.vn/thanh-tra-kiem-tra-kiem-nghiem-san-pham-duoc-tiep-nhan-xu-ly-cac-thong-tin-ve-thuoc-gia-5044678.html',
  'domain': 'baolangson.vn'},
 {'id': 67234, 'url': 'https://baonghean.vn/an-uong-lam-am-co-the-10014055.html', 'domain': 'baonghean.vn'},
 {'id': 80524,
  'url': 'https://baophutho.vn/4-mon-pho-bien-tren-mam-co-cung-ram-thang-gieng-co-y-nghia-mang-lai-may-man-248768.htm',
  'domain': 'baophutho.vn'},
 {'id': 84125,
  'url': 'https://baoquangninh.vn/tap-huan-cong-tac-giam-sat-phong-chong-dich-benh-do-vi-rut-nipah-niv-3394596.html',
  'domain': 'baoquangninh.vn'},
 {'id': 88645,
  'url': 'https://baoquangtri.vn/suc-khoe/202207/50-benh-nhan-covid-19-dang-dieu-tri-tai-benh-vien-2202255/',
  'domain': 'baoquangtri.vn'},
 {'id': 90572,
  'url': 'https://baotayninh.vn/hanh-trinh-tan-hien-cua-nhung-thay-thuoc-uu-tu-140074.html',
  'domain': 'baotayninh.vn'},
 {'id': 91319,
  'url': 'https://baothanhhoa.vn/bac-sy-benh-vien-108-da-thuc-hien-thanh-cong-300-ca-ghep-gan-267964.htm',
  'domain': 'baothanhhoa.vn'},
 {'id': 97029,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202511/dung-lieu-an-thuc-pham-moc-nguy-hiem-khon-luong-5310d8b/',
  'domain': 'baovinhlong.com.vn'},
 {'id': 98260,
  'url': 'https://benhviennhitrunguong.gov.vn/thuoc-diet-chuot-va-nguy-co-xuat-huyet-o-tre-em.html',
  'domain': 'benhviennhitrunguong.gov.vn'},
 {'id': 98535,
  'url': 'https://benhvienvietduc.org/chuyen-gia-benh-vien-huu-nghi-viet-duc-tu-van-benh-ly-di-dang-dong-tinh-mach-nao.html',
  'domain': 'benhvienvietduc.org'},
 {'id': 99635,
  'url': 'https://dantri.com.vn/suc-khoe/dung-thuc-pham-chuc-nang-nhu-the-nao-de-khong-loi-bat-cap-hai-ky-4-20251009161639294.htm',
  'domain': 'dantri.com.vn'},
 {'id': 108128,
  'url': 'https://giadinhonline.vn/giup-me-bau-ngu-ngon-voi-10-meo-don-gian-d19025.html',
  'domain': 'giadinhonline.vn'},
 {'id': 118141,
  'url': 'https://hanoimoi.vn/dung-de-thuc-pham-ngay-tet-thanh-ganh-nang-cho-gan-ruot-734057.html',
  'domain': 'hanoimoi.vn'},
 {'id': 125814,
  'url': 'https://hellobacsi.com/nuoi-day-con/nhi-khoa/van-de-nhi-khoa-khac/tre-co-ham-rang-bi-o-vang/',
  'domain': 'hellobacsi.com'},
 {'id': 140282,
  'url': 'https://khoahocphothong.vn/phat-hien-khang-the-co-the-ngan-ngua-hau-het-cac-chung-hiv-240586.html',
  'domain': 'khoahocphothong.vn'},
 {'id': 154292,
  'url': 'https://laodong.vn/suc-khoe/4-thuc-uong-giai-nhiet-giup-giam-trao-nguoc-axit-1205578.ldo',
  'domain': 'laodong.vn'},
 {'id': 176969,
  'url': 'https://medlatec.vn/tin-tuc/bi-mat-cam-xuc-khi-quan-he-co-nguyen-nhan-la-do-dau-s78-n22073',
  'domain': 'medlatec.vn'},
 {'id': 199047,
  'url': 'https://mega.vietnamplus.vn/thai-huong-nu-anh-hung-dam-me-lam-cho-dat-no-hoa-5415.html',
  'domain': 'mega.vietnamplus.vn'},
 {'id': 202493, 'url': 'https://nhandan.vn/post-853313.html', 'domain': 'nhandan.vn'},
 {'id': 268030,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thong-tin-can-biet-ve-phuong-phap-phau-thuat-lay-duong-ro-luan-nhi-59283.html',
  'domain': 'nhathuoclongchau.com.vn'},
 {'id': 304343,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-loi-ich-bat-ngo-cua-nam-oi-voi-nguoi-benh-tim-mach-c12a511869.html',
  'domain': 'phunusuckhoe.giadinhonline.vn'},
 {'id': 312382,
  'url': 'https://phuyen.baodaklak.vn/95/141195/mit-tinh-huong-ung-tuan-le-truyen-thong-ve-phong-chong-khang-thuoc/',
  'domain': 'phuyen.baodaklak.vn'},
 {'id': 319666,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-amitriptylin/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io'},
 {'id': 459741,
  'url': 'https://suckhoecongdongonline.vn/tu-vi-thang-5-2021-am-lich-cua-12-con-giap-chi-tiet-nhat',
  'domain': 'suckhoecongdongonline.vn'},
 {'id': 546923,
  'url': 'https://suckhoedoisong.vn/tai-sao-ngay-nao-cung-can-chong-nang-cho-da-16995354.htm',
  'domain': 'suckhoedoisong.vn'},
 {'id': 565289,
  'url': 'https://suckhoeviet.org.vn/cong-dung-tuyet-voi-cua-hoa-dau-biec-voi-suc-khoe-va-nhung-luu-y-khi-su-dung-8483.html',
  'domain': 'suckhoeviet.org.vn'},
 {'id': 570069,
  'url': 'https://tamanhhospital.vn/tin-tuc/cuu-song-be-so-sinh-bi-thoat-vi-hoanh-the-tui/',
  'domain': 'tamanhhospital.vn'},
 {'id': 651969,
  'url': 'https://thanhnien.vn/tong-hoi-y-hoc-vn-de-nghi-cho-bac-si-vu-tai-bien-chay-than-duoc-tai-ngoai-185676141.htm',
  'domain': 'thanhnien.vn'},
 {'id': 660759,
  'url': 'https://thaythuocvietnam.vn/hen-phe-quan-va-nhung-dieu-khong-the-bo-qua/',
  'domain': 'thaythuocvietnam.vn'},
 {'id': 674464,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/vi-khuan-nhuom-soi-la-gi-y-nghia-trong-chan-doan-nhiem-trung',
  'domain': 'tiemchunglongchau.com.vn'},
 {'id': 678742,
  'url': 'https://tienphong.vn/weilaiya-white-truffle-2-nam-lien-tiep-la-san-pham-dau-goi-tot-nhat-nam-tai-elle-beauty-awards-2025-post1735881.tpo',
  'domain': 'tienphong.vn'},
 {'id': 681131,
  'url': 'https://tuoitre.vn/loan-thi-truong-affiliate-ai-chiu-trach-nhiem-chat-luong-san-pham-20260130000926371.htm',
  'domain': 'tuoitre.vn'},
 {'id': 686659,
  'url': 'https://vietnamnet.vn/cach-dap-mat-na-cham-soc-da-mat-phu-hop-voi-tung-loai-da-2018837.html',
  'domain': 'vietnamnet.vn'},
 {'id': 702221,
  'url': 'https://vnexpress.net/bo-y-te-virus-nipah-co-tiem-nang-gay-dai-dich-5041215.html',
  'domain': 'vnexpress.net'},
 {'id': 710247,
  'url': 'https://vov.vn/suc-khoe/nhung-thu-co-the-can-khi-ban-gia-di-post981896.vov',
  'domain': 'vov.vn'},
 {'id': 717975,
  'url': 'https://vov2.vov.vn/suc-khoe/phong-va-dieu-tri-benh-viem-bo-mi-mat-24968.vov2',
  'domain': 'vov2.vov.vn'},
 {'id': 720463,
  'url': 'https://www.pharmacity.vn/tang-sinh-mon-sau-khi-sinh-lieu-co-nen-khau-ngay-lap-tuc.htm',
  'domain': 'www.pharmacity.vn'},
 {'id': 722587,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/covid-hom-nay-12-5-them-2-439-ca-mac-moi-1-benh-nhan-tu-vong-727956',
  'domain': 'www.qdnd.vn'},
 {'id': 724965,
  'url': 'https://www.sggp.org.vn/ho-tro-con-dao-thuc-hien-phau-thuat-noi-soi-co-ke-hoach-post817576.html',
  'domain': 'www.sggp.org.vn'},
 {'id': 732544,
  'url': 'https://www.vietnamplus.vn/to-chuc-y-te-the-gioi-dua-ra-khuyen-nghi-moi-ve-dich-cum-gia-cam-post942110.vnp',
  'domain': 'www.vietnamplus.vn'},
 {'id': 750899,
  'url': 'https://www.vinmec.com/vie/bai-viet/roi-loan-tieu-tien-o-phu-nu-mang-thai-vi',
  'domain': 'www.vinmec.com'},
 {'id': 760817,
  'url': 'https://youmed.vn/tin-tuc/chi-nha-khoa-va-tam-nuoc-dung-cu-nao-tot-hon/',
  'domain': 'youmed.vn'},
 {'id': 766444, 'url': 'http://ask.familydoctor.com.cn/zt/mn.html', 'domain': 'ask.familydoctor.com.cn'},
 {'id': 772903, 'url': 'http://baby.39.net/a/141222/4540983.html', 'domain': 'baby.39.net'},
 {'id': 794284, 'url': 'http://byby.39.net/a/230524/13823332.html', 'domain': 'byby.39.net'},
 {'id': 801009, 'url': 'http://care.39.net/a/170224/5230598.html', 'domain': 'care.39.net'},
 {'id': 812548, 'url': 'http://ek.39.net/a/20101018/1506617.html', 'domain': 'ek.39.net'},
 {'id': 818281, 'url': 'http://fitness.39.net/a/180124/6017569.html', 'domain': 'fitness.39.net'},
 {'id': 836246, 'url': 'http://fk.39.net/a/241010/i1trnyn.html', 'domain': 'fk.39.net'},
 {'id': 836542, 'url': 'http://fk.99.com.cn/news/410997.html', 'domain': 'fk.99.com.cn'},
 {'id': 844242, 'url': 'http://food.39.net/a/2010821/1451769.html', 'domain': 'food.39.net'},
 {'id': 852556, 'url': 'http://gan.39.net/a/260621/x3wnanf.html', 'domain': 'gan.39.net'},
 {'id': 853240, 'url': 'http://gc.39.net/a/141221/4540391.html', 'domain': 'gc.39.net'},
 {'id': 854825, 'url': 'http://gk.39.net/a/090624/900447.html', 'domain': 'gk.39.net'},
 {'id': 857643,
  'url': 'http://health.people.com.cn/n1/2026/0701/c14739-40751470.html',
  'domain': 'health.people.com.cn'},
 {'id': 860914, 'url': 'http://heart.39.net/a/190801/7350145.html', 'domain': 'heart.39.net'},
 {'id': 868618, 'url': 'http://man.39.net/a/220924/8244035.html', 'domain': 'man.39.net'},
 {'id': 871353, 'url': 'http://nk.39.net/a/150617/4640167.html', 'domain': 'nk.39.net'},
 {'id': 876479, 'url': 'http://pf.39.net/a/2011418/1666222.html', 'domain': 'pf.39.net'},
 {'id': 895407, 'url': 'http://qihuangzhishu.com/764/120.htm', 'domain': 'qihuangzhishu.com'},
 {'id': 900894,
  'url': 'http://qy.familydoctor.com.cn/a/201405/609715.html',
  'domain': 'qy.familydoctor.com.cn'},
 {'id': 901143, 'url': 'http://shen.39.net/a/140114/4326874.html', 'domain': 'shen.39.net'},
 {'id': 904470, 'url': 'http://sj.39.net/a/140814/4447769.html', 'domain': 'sj.39.net'},
 {'id': 905348, 'url': 'http://tnb.39.net/a/130607/4186274.html', 'domain': 'tnb.39.net'},
 {'id': 909349, 'url': 'http://wei.39.net/a/110821/1781775.html', 'domain': 'wei.39.net'},
 {'id': 914722, 'url': 'http://woman.39.net/a/140915/4472540.html', 'domain': 'woman.39.net'},
 {'id': 1826763, 'url': 'http://www.cnkang.com/myjk/201202/480566.html', 'domain': 'www.cnkang.com'},
 {'id': 2040931, 'url': 'http://www.wujue.com/zyzy/zycd/p/200704/14378.html', 'domain': 'www.wujue.com'},
 {'id': 2065346, 'url': 'http://www.zydcd.com/zhongyi/zy20700zh/', 'domain': 'www.zydcd.com'},
 {'id': 2131080,
  'url': 'http://yanglao.familydoctor.com.cn/zt/scsjtfz.html',
  'domain': 'yanglao.familydoctor.com.cn'},
 {'id': 2136074, 'url': 'https://article.iiyi.com/detail/416303.html', 'domain': 'article.iiyi.com'},
 {'id': 2211756, 'url': 'https://ask.39.net/question/20931974.html', 'domain': 'ask.39.net'},
 {'id': 2475161,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201511/856792.html',
  'domain': 'baidianfeng.familydoctor.com.cn'},
 {'id': 2501085, 'url': 'https://bingli.iiyi.com/show/64998-1.html', 'domain': 'bingli.iiyi.com'},
 {'id': 2507808, 'url': 'https://cancer.39.net/a/210407/8822699.html', 'domain': 'cancer.39.net'},
 {'id': 2516720, 'url': 'https://jb39.com/jibing/ZiGongNeiMoZengSheng264317.htm', 'domain': 'jb39.com'},
 {'id': 2525727, 'url': 'https://jbk.39.net/shoushu/xws/', 'domain': 'jbk.39.net'},
 {'id': 2541313,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000EO44K',
  'domain': 'test.pmphai.com'},
 {'id': 2551069,
  'url': 'https://v.familydoctor.com.cn/a/201509/816823.html',
  'domain': 'v.familydoctor.com.cn'},
 {'id': 2843944, 'url': 'https://www.120ask.com/question/38159402.htm', 'domain': 'www.120ask.com'},
 {'id': 2992495,
  'url': 'https://www.a-hospital.com/w/%E4%B8%8A%E8%85%94%E9%9D%99%E8%84%89%E7%BB%BC%E5%90%88%E5%BE%81',
  'domain': 'www.a-hospital.com'},
 {'id': 3160976,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8424738550787796805?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com'},
 {'id': 3530740, 'url': 'https://www.familydoctor.cn/a/201309/512842.html', 'domain': 'www.familydoctor.cn'},
 {'id': 3891869,
  'url': 'https://www.familydoctor.com.cn/tangniaobing/a/201702/1706297.html',
  'domain': 'www.familydoctor.com.cn'},
 {'id': 3980742,
  'url': 'https://www.msdmanuals.cn/home/skin-disorders/nail-disorders',
  'domain': 'www.msdmanuals.cn'},
 {'id': 4013480, 'url': 'https://www.youlai.cn/yyk/article/608596.html', 'domain': 'www.youlai.cn'},
 {'id': 4030514, 'url': 'https://ypk.familydoctor.com.cn/267067/', 'domain': 'ypk.familydoctor.com.cn'},
 {'id': 4065823, 'url': 'https://zhongyibaodian.net/a/142698.html', 'domain': 'zhongyibaodian.net'},
 {'id': 4182981, 'url': 'https://zysjonline.com/articles/misc/16204/', 'domain': 'zysjonline.com'}]

SOURCE_PROFILE = {'rows': 4394718,
 'id_min': 583,
 'id_max': 4420561,
 'duplicate_ids': 0,
 'hosts': 97,
 'host_counts': {'special.vietnamplus.vn': 1,
                 'bachmai.gov.vn': 2783,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'dantri.com.vn': 2063,
                 'giadinhonline.vn': 16729,
                 'hanoimoi.vn': 1283,
                 'hellobacsi.com': 11747,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'youmed.vn': 7189,
                 'ask.familydoctor.com.cn': 3,
                 'baby.39.net': 27621,
                 'byby.39.net': 439,
                 'cancer.39.net': 8415,
                 'care.39.net': 15134,
                 'ek.39.net': 3222,
                 'fitness.39.net': 12066,
                 'fk.39.net': 9812,
                 'fk.99.com.cn': 1,
                 'food.39.net': 11024,
                 'gan.39.net': 5027,
                 'gc.39.net': 1746,
                 'gk.39.net': 2059,
                 'health.people.com.cn': 1348,
                 'heart.39.net': 5965,
                 'man.39.net': 5320,
                 'nk.39.net': 6650,
                 'pf.39.net': 1538,
                 'qihuangzhishu.com': 23637,
                 'qy.familydoctor.com.cn': 40,
                 'shen.39.net': 3223,
                 'sj.39.net': 942,
                 'tnb.39.net': 4178,
                 'wei.39.net': 3894,
                 'woman.39.net': 11687,
                 'www.120ask.com': 918479,
                 'www.cnkang.com': 963438,
                 'www.wujue.com': 53341,
                 'www.zydcd.com': 78680,
                 'yanglao.familydoctor.com.cn': 2,
                 'article.iiyi.com': 5641,
                 'ask.39.net': 337375,
                 'baidianfeng.familydoctor.com.cn': 3798,
                 'bingli.iiyi.com': 24990,
                 'jb39.com': 7306,
                 'jbk.39.net': 19142,
                 'test.pmphai.com': 15120,
                 'v.familydoctor.com.cn': 10,
                 'www.a-hospital.com': 169339,
                 'www.baidu.com': 2631,
                 'www.familydoctor.cn': 44,
                 'www.familydoctor.com.cn': 447453,
                 'www.msdmanuals.cn': 6173,
                 'www.youlai.cn': 46119,
                 'ypk.familydoctor.com.cn': 1,
                 'zhongyibaodian.net': 146700,
                 'zysjonline.com': 243347},
 'first_1000_host_counts': {'special.vietnamplus.vn': 1, 'bachmai.gov.vn': 999},
 'sample_host_counts': {'article.iiyi.com': 1,
                        'ask.39.net': 1,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 1,
                        'bachmai.gov.vn': 1,
                        'baidianfeng.familydoctor.com.cn': 1,
                        'baoangiang.com.vn': 1,
                        'baocantho.com.vn': 1,
                        'baochinhphu.vn': 1,
                        'baodanang.vn': 1,
                        'baogialai.com.vn': 1,
                        'baohaiphong.vn': 1,
                        'baolangson.vn': 1,
                        'baonghean.vn': 1,
                        'baophutho.vn': 1,
                        'baoquangninh.vn': 1,
                        'baoquangtri.vn': 1,
                        'baotayninh.vn': 1,
                        'baothanhhoa.vn': 1,
                        'baovinhlong.com.vn': 1,
                        'benhviennhitrunguong.gov.vn': 1,
                        'benhvienvietduc.org': 1,
                        'bingli.iiyi.com': 1,
                        'byby.39.net': 1,
                        'cancer.39.net': 1,
                        'care.39.net': 1,
                        'dantri.com.vn': 1,
                        'ek.39.net': 1,
                        'fitness.39.net': 1,
                        'fk.39.net': 1,
                        'fk.99.com.cn': 1,
                        'food.39.net': 1,
                        'gan.39.net': 1,
                        'gc.39.net': 1,
                        'giadinhonline.vn': 1,
                        'gk.39.net': 1,
                        'hanoimoi.vn': 1,
                        'health.people.com.cn': 1,
                        'heart.39.net': 1,
                        'hellobacsi.com': 1,
                        'jb39.com': 1,
                        'jbk.39.net': 1,
                        'khoahocphothong.vn': 1,
                        'laodong.vn': 1,
                        'man.39.net': 1,
                        'medlatec.vn': 1,
                        'mega.vietnamplus.vn': 1,
                        'nhandan.vn': 1,
                        'nhathuoclongchau.com.vn': 1,
                        'nk.39.net': 1,
                        'pf.39.net': 1,
                        'phunusuckhoe.giadinhonline.vn': 1,
                        'phuyen.baodaklak.vn': 1,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 1,
                        'qihuangzhishu.com': 1,
                        'qy.familydoctor.com.cn': 1,
                        'shen.39.net': 1,
                        'sj.39.net': 1,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 1,
                        'suckhoedoisong.vn': 1,
                        'suckhoeviet.org.vn': 1,
                        'tamanhhospital.vn': 1,
                        'test.pmphai.com': 1,
                        'thanhnien.vn': 1,
                        'thaythuocvietnam.vn': 1,
                        'tiemchunglongchau.com.vn': 1,
                        'tienphong.vn': 1,
                        'tnb.39.net': 1,
                        'tuoitre.vn': 1,
                        'v.familydoctor.com.cn': 1,
                        'vietnamnet.vn': 1,
                        'vnexpress.net': 1,
                        'vov.vn': 1,
                        'vov2.vov.vn': 1,
                        'wei.39.net': 1,
                        'woman.39.net': 1,
                        'www.120ask.com': 1,
                        'www.a-hospital.com': 1,
                        'www.baidu.com': 1,
                        'www.cnkang.com': 1,
                        'www.familydoctor.cn': 1,
                        'www.familydoctor.com.cn': 1,
                        'www.msdmanuals.cn': 1,
                        'www.pharmacity.vn': 1,
                        'www.qdnd.vn': 1,
                        'www.sggp.org.vn': 1,
                        'www.vietnamplus.vn': 1,
                        'www.vinmec.com': 1,
                        'www.wujue.com': 1,
                        'www.youlai.cn': 1,
                        'www.zydcd.com': 1,
                        'yanglao.familydoctor.com.cn': 1,
                        'youmed.vn': 1,
                        'ypk.familydoctor.com.cn': 1,
                        'zhongyibaodian.net': 1,
                        'zysjonline.com': 1}}

SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
CRAWLER_SHA256 = '9d32b5b911ca8ac6b53331e251e548a914281b9e794cdd87ee56451e6b28f6fa'
CRAWLER_PAYLOAD = 'c-qZ<|C1cWmFMq7@IMqH9Mugyr5=6ZV5G(?gs>HW;Dqqq#i-q`>8_qGrn{=Cs-7>*%nx3N_iUfJFt^?f9I+b<<~G)0uMNA$`3|&q{y_x)haxP%|HXaYdzo38)m1$s9T&Ikdel{w`7-n6%a<?Tw>)#^%y;7F;`HU~H@#sN?oHRyWP0Gu<7rw1-i_3Ia}i|+-Zb2brg_Jkr6c~g$ig^@lWi}}yf_&}`}iH4IdkUGN5)w?^9F<QqF7|nVBp2Gd72eom?UWt7IB*7k3RC~Bl>N&JrA=yQq_j(bQ%r0kgj(%T_i=6b-YnD4j0p66b}niIvFmqEJ}(%S~HbyO|mE)y_%-eYx~h~QKYCJ=7V86n@^)68kGQwXup`oTe_lpGYgY&JIYkCQCLJpJd1Q8{R}MN?^|i2yC2QMcp5B<cxrmWqoWFgA|2?8X87B0#dGs83G)fMVxHw6ya{Ge5u$Ebewv4)5yqRFH*e%=VjgB;F;QKb<EJ&gSNV&2$iifl&dSHKU-^UvQ9h=_ov0`uX&&<;9+u7L2j%aAkq>$(-`HbbB=HbLB;(;eF0v_&l2N5Qig$0Manhk5`Ft7|ZrLo|N(-)#=^8ib)eG@7dX-;WL^eMNvn<^M1ppn@w*e8wqKNqGaOo@Bzyc`v>zfv;@G>u~F=yd8o`xVoJ*Z+j(mmUbiotj~q8@$c+RfLlzjDLtdH(JN|ItTYzd}Ft0=yaJQJ4)US7{wa+1YNejxsl2dG3{4gDc;<cH`EH!ajk2^bxEQZ&3_e?Ttqt@$g?3p-+h+*&s>xTAFG>sl#><=jk}jKr*d%31Sp)V|*=|qzWWOr9nOk&s}^<N(7T=U)8W+03p$UinoXeHJH7)m@pa2+aR4sNz2c+{5IiutV-21$lz2f<BsP`2mFS{!y=u<!@<@;5#`)c$Fm1RZ-C6QALVPBa$+g7ol%^%<RS0f$`)AfU~_pfNO$;A)hVLc9H6)=4Q`NxvxtEP^viqF8~eebSrh?LLp%b|R{J<8W^><y6YOO%ScUYO`)hs6qWLr&M!LUxq*8^}zZz${^@bj14Hl?iFj~y!c?$(Gq$DRk9p=M0?!6FBa}1LhdQ$YxwSy?3wc7F*#dz&$zil-;O~X;yYK2I+5h{_ltf?jp42Zq97%Y&{s{u@Pr>e79f%%HD_<lf4qGgRKs~b@pli=S@d=G@~(S9X>SB!3wf}Fr>D&uNtZp7aLD#ddQus$epn(jrJO@#?9tZDpK)S_W)VDuXlV==;@Ea=~O;i}(}G77Aoc>i{WL3DU1&RhM?0iQ)dzSwGI{^sq%yS?ak!*2gcD(NZkZD(kv`0eEIeCJp|n&t%0`$1l0@x0Y$MD7zJ2~Z#?;V!h>x=e|MIl78v)WT+OW;0zBdi~Ms0MDvF@z`t8*R!6FZ8#odVFxt97&0%v-0(K;eHdfY_)a)oMAyKuvzGtLq5wbl!YPs40kID+^faVP80Ug^8Ak4KlBReFy(HT6*4EH!pplrTi)<J<U6favQ;ccNzZtwL`vdJeEvEp6F&_u{Z~~!VAbSasO71<@^EUl>M2H0u`F-jLL?bYepg5REIT-RFh7<z%qg914j-%<QMYY>pcr_-j?av`hMc9V~oEMKEhK)R$AV_X#odmAMEXmEV<54c-lC>IcnPGKw*&g@4-<E&OU{Nzj7Skz&EC_5gsQED9-(cDh7LsAnLYZ~|Ic$Ca5&|BkVEy!N<tSc`^IQp<qXh_Uh^<Ud&-Lfau5(+|dnGZp)cbLqfoC^;ecXHf=1X3<C?+ZR=z$mKUcMNQ<6#UA$YegxQfxiw!6k1OOO$p}Z;XK+cv}&^;~^@B6XNWuJ=EC>@rsrQCJ@{?8}JM8trCYk_x8xC9!&{qap^$Hwz`<=6eyd<w8L2y6yk|?VqgeTSps$d5FMJ~wx%mA>>0ts(*EwGDCYS$H_o2j{O(z11{RC6|67`cv41_`(QC{DYa8Ansy~(;>lx7hL8!#xfH+(>)x6;>8av<KJl8Loq+La(XH#nSy(fF#`qDu2#jIsdX0S{B13PGYm%a6Fw_7t1yL=6FK$WEp&g(tPEiAzX8p|?hXeF^4EQNH^@TiU<(8s-NZ!W^=8tocjPt$bB$03o3vWQkm7H=*19T5YS%jN4B{E4<KMPhF*aFham=`0kGw7C-<<XEQ}7P}fiyW@3LyL*$Ex`MHBJqK-OWEDbGQ;7fPI?GhYuMI{oS0|9!%&A+{|G7ZRY09wtd+N9n+~j{0OOB$E*~TFB@1~e<LQHv9Y|*;WOg)N{8V0dy!M2wWGlcZk(JK|_x7Lc)+Hsfiy7RHR@E&5>(12DfnmG=@ED31T=(29HX*`Q*H+Im!G~>jq2_Q`(*Ql95`W}RHi1ee@A?*(v24T$bHauoezO=z#>Bn(pSkN%kkq7G;XguJxL=5=fHnt}V2kn|lV^y!Y*ShDMT(FN89;EMzv}D2eJhcj2nQR_310-*nDKLwn6h;}V9S@7RHw|Z7qtMGXyi9Gg3Vab}1@ZNi&KfvEghi3@8bFWY(SW~$R&L-}2T*kd(z*U#%2`~E_B?)1TXKt=&~H}qy4N6G#A9rODLB;HT0B}4aJ2T-BS5vH{o!<x$GcIxlrqMiX_>YtYRhUWkESBGG)W@ZkFiFvi?);?^=!}Y2uKWGzr5@V!rFaj*;(yW=j`x_Co+46`4o`pz52MEG2O!xX3bhT(iWq*i38Scy3e!hmVstikUg|O9rKPrXHq5Qt*!AEb!gxp2lmdaH;qFX-iFt$aC%|k$CVnvVooAYOEXQ3P~oR<;X}IAHnRPxPtS7NQ}VumVQGI#lP6HsE><3Mb!n!Cfyif43zZJb{`-U=Oe1)@)}TQ|oeczUv?^vi%@9=a!;wza=Wt(r4)@*9RwG6z1*@Rex~R)rH-6J}nf;jGDp>^I<cAGOgcXux25wPXis@)HRm$*WDQL%yICD=_F1ChPX`rJ)SP-ux9TKetAT5vRjeq6(w-(vr7Sz7`XSEWF)){oB<9M4&4AX4B$mtz#U-XLg#J)XBnKpcbxT97y)_9&ycO&eH9W$>=zq4WKk<<apfkwE1@J(%D#`J3OYk3TBES?{c7$)d5eA(%uW~czCco%n<S+HiN1H;$&TRNwnLma`A5l#4&a0*2XilmoDP&?eZRbe4$6?MCG09;gd1U>~1vPGiz1dA8pxV{^wi`+T<nhI@OL5Fw5r8oPokVMK>!xbuAE}b%8*KM4ozJPA9bntt8<`8v-haU5Y&(xlG0HjVCP@X0d95@$Ig}-t;XuMQi0Zx0V0VyY)AbxFnL`MPrQI78VD?gp)9Od6+^~3e&&(g7T7Uz643;d>DkhT!o(Qt1-8ZXk3Vsky$?RuAcP6;~L5zQ97{JJEgov%F*0hEh=Js{IYf4S0D3Pnnftrfv2B855G9$Kg(yR+!z+V#?x(#;YK-u~<c718$_gt8{|(I9ArG$T;t=^;7|{ZtkXrTva+ob5bNO^_1B@0ah#aRSyL`+W`a1mkK)1pE`r5T}7!fy>qu9#6yV0UiBN3z7y`ttkf1gIUI4gd?10%B0GJ>av})1>1mdbeSIzOQPBCD6l6ETR6jjj;1|g^P1iqMh~rEmJUHkyyx_I$Ekt5yop&1gBSs+)$|P3_Kf<XPbK!^whGrZLJV(dzckEuk*F=Kc5J@FfQSV0pt<oJ+PA{iSsym-5>0{xx~Li_wdpZljj<I?^i0zK4X0tAdu$t$rE_h67|ltypcy7<W(U~e;$kqsA!<5i2^tK9OsxeBnLX&9U8x+}NG?GlGykx@F6MMR4GctqK`c{>I{kWqRAy1u+!Kt8x-kt8dg}osD(qb_fq-_;FLjBF@-Hs0e(MT)6C*?^h$XZNr0qq>`$K7~c?pJ3=fpt8pt#w|RWk|L3y4=eu~jl3*ahSjiQUH?>wn82!>Q9UzY!(6MiGG>V0%?4%A#%37#;8t#KsD1jLHF3PZ?U>FSQ}36iQpt0-Vm$;GHhySgjx-?t$H;+LpnSS;#2I#1atK@v9qkStw9NSXCj0)!;SNQv0MzF#m-k90oP(IrLwLNVj%n8>|kC<n=4)!j&P>K8vH6ud|b4Htm2Pu-Fa|XXm5wr8j%s;F-=7XP@9-7yaY*k|lDvagCclW0Uw0a@ZstmFUv?;#Nh?Doz2S=_$=g4mzcXX#XS8@xuJX`#j0ltjP7-rR{)A+x95+3XCVKyqaSL?xo9w!JDncG5lDRnY<m=_KvOFx^E;#cD$#tST9^S-+mzA7!Ycc0((R&kigmvKJHz?WF2@#s<qD~1x35$m!daMxT??#QO0{bNoT=mcS&0hzCVZgs?h3FEvACy&sa`n_XZNA$g3hw#%Zg9(2g@=RBd65%hS~B>6wv}#?#$)lfqs*a0KyupnkTfKbsf&o83NH<p<l-bSs=TN|iP;7fBwDBMr#!R0+aa#A=NVQZg3^FJa9VtH(6gtsV-P_8oe*zM<oyl_EC>u|SN${0BMYhNPGt$j0O;opz;m$*vp*XV@%&SCPn`4EZ#Q<}EszcU@-85;0X{L3m9{&r+a0g~b&dt-@2l1#6HjhsWMh?HU2a3_!ETHg|#XK-IDL^jc4MAh);KG!un0C4-R_V9yAExwckA(}Q`OV@X5}lJRh|NC-Ox4*gR;fnl)qMaOLDL~*5uWU!*-R#I^5sf!oSKh<t%ic!SPJSi4C2G(va3=)^}&#u(CHdAwog-rS@(&CN%m&wLfOu_`(F9LV$l}8XTQ9gV4Cv>MWhad`>-e<iF7azEL8i#`sb{^;+qE)TSL}}@B73#F`rS`LqK6RNXlL7<s8R(Q7CFp4}WaqcwDl|3MQ8EGXLoe}s%x4N!TY)8$gm#;dzBP1b2&MEW-J?yJ)|f3|@jtYhyA5?k_d<dqZ44LAJ=5_ncF(UOFjk7&gVnn_pKfnPSfguK#-Lf+E4r%?UcZ`Gj7&e05S_vcio4QY(L%(Mv~?^!p}muPw6a&V9)zlw0sRK5XY-cfbo_Z$|E)2!%0YCzTR01=pRZsA0vmh$*gi$?Nv3EZYl|~)5?AimFje^H=m9rq0+a++w?^C=e1f0Wa`d{M7}~DeJUVq9U9lS6ZI&ee!90bOc~EY%ju_Z10m2z!n{9z(|8TCvi(}aB@sLCXw#H-s<Q>#KjtIyac|^6P<T$Hlo#?@XBI9RI*Mn=-St%bB`-Q(;2ghT$ld&_+p|YeIFj&nju{mnz;~I5F;~G*Pz{EQ=(B^$`z<VH@4Oi}WO}to;W6Mvf`3@Wq&Ry`H;D1(cMiys0-(81!w|kEMoToqkhW<P)T0VyZzNDN5{F4@Ai4NsKTlP=^+$@3h+b40#Iz6hR8H6yiQ~y~{yFQr7xO*D4M$^_a9V8?xm0-`-8aD4TN)xJ3+NAR|L?n)J1e4X6rRyInY}lRd1WCfbOovBbd%kUMkhR7b=h<QJ@?%i;v0kyE)PoBgvW~G?!+V!K^1glR<(E8i`N5QWsp2)V{#Nu~co8sAag}bFm2+Zcanf#;UVkM2<SVxuO$VEN`1YCo<3|J0Oe?ZO_X5t7>)lmI`}&5p*|Z>*yM={&VYNupIFG<atYllI9s2gITd#VD0{_@=>_g*e;FM(ZVC`8O!KSc!n!1)CtYfCt@hPYgF^t}kPBnPfFocNu%<B4~Rx*<3M(Njv4;^cGZDq@=401Iq+Vr^@ra{u40%m)2<097P1ED}P9VG%7=xWWi)>N`)nE{gax}~_Bz}D`_ZFW~yKCfjv?W%82bsjVdYkKV{Z=xERb%}Z+5d;kr=Uqs!rw80#(*LQ&Gz^mg@9HfpbfMPQuz8KdIiHProwzGl<gK1*j4BWKrQIC!;;!w!!a537NpIow)~V$TiR;i-Oh0s;s#K)ALt>$l2)THU4JH|}h0U(LtW{K%G^TA4Qjln=Xvmc1XhWRs-g-R|PqM0E){=joZr2T@;~9l@Nk&?TRgEpFPeNK*imLls5UU&qsCA`wFZ-}wRn|Z0t=F+sX!dxFT6{DTgRg3m(OrJ=yOn~Z^j1xW-`7WTZlnDmlIKGSiAqFN|ESYL6UV16Z92ByevRp+dZsg~ft&KfaA2J4UQ^6-H7}NS>v-k)MqWv-v}%5PJ$G6kXxGB=2FmH3f#x+XpN7{Q2iolq>tR}&!E@ddPv|Z+>NuL@+v;`??VVC%bm`AIR%p<l3nHxRp<h1L$RGD!O5xH<t|w4g>yt0y6rkzCy2sF=*n+k@MA#*C9&FnbZTBJx8i>)=!5WHNrH8Fw&o&AkK&uK}+Ed*JlTS#e!74qR{`9a(y0z3jOw<wBD959Edm$WEw-zJxrAci0x+tLLWW&5eA7`vpx#~<)>UNnYcRbtkpy!<NuF97+(!&rMeA`W^MkAuMkP1fcaA(8YWj?#p!4J@_hE8?`ayZ3|?wF$ro0C*$)(X`mEFjzTFO0Os8bW>Ju}G@^)SN$$mt9r9{u8ZnczDP|EaSR#L?)i$Y#T+2M`?sCl)_s#)Ig5w0GfSPYMtKai1&C`O;x8-7AleD;M;+i=}d*pO`!rxObi)9tzQI<Yl9dcZ3)L)r0qd_3o@m}k;D&>;0gUXG~V%W8m+-Ge-?t9Ork}`al8_^t*8o?U`ujfcx=D++3+}ucbOhywnDD+-*f!;JpaAGe=qXir|56ND%)J5y6GoFByYvYKemOPT?qR{D}e-TQ^e!mbbm&1N;4XL4iob(1#H2*Hzk)~Ob6m}UZH@Uh^Al^Q!}Aefp{v(rfTpKu(l+*S_cj4K>)B4MN?}%FW;e@+8=eb_OhowSs+l3S6n_sa9qJCDGsK*=;Cn(6CM3ZQtR_BuS5#X^-t<W*?^iZFahov?2%iUzf_y%Sg#H%aM0j`MglagtY=EKT@e$ZP!#!`FqjGinPM)n%tiQhKu#lddPk@2vS<UA3uap6Ft>W_-jV0GoAG!W#7PkunO>t#V5Jq>sq8KS#|}4joz(0t!Z|jff2_P><5f;8yS3$i<JIS1SQA$Ozv9&3i(7XItwsf9;@xN1*v1v5vU)2)+RAb%eI%DK-okuphXqRfs32^PBnrekHcK4PD-MBXrbdqUB9urt1byX*h|#!T%SaFX0y-yhxutFyzgON-<clDJr=QeK8#ewqpjGC8yVh>nVSP0rkU-%yqOd?Pshap)E2k`hLt7_=jDrC`(oTZdC;lv+m@+sX8J9mc^XZE|Q7PZ+=1SPQRy?YLGzps<=N#puDyP+t-bocUR_#deC|Jzr92YTgoF!fIG@)&;W+cGDPD>IjoV-Ya#V!o38e(ou!U!;C_=U+{K+c=wY6NbMf2Q8Io;!Db{hW{hsOjoK79-WeQ1NO7OT{&F;sXRUs<*fECwlnb=It}L{r)#wo45DY`cJm~>PkY*>!n9DAx72Ta8Aw-(T-qu{i*X$U+5dvG6$eDgWm=XMt<ArfewL3OA30hou!L81&J9MZEX6(+O8X%YDBAXGyvv!+gpS^YHfRKhyK}LqaXGpTP>o{`qSO>U8OH0N`e2~e4*9jdRL}_4EDk-k!tLou+#yoiRHuc5@?McU7L5p45n(E5$mJu>k=aWJXcY+$(Pb}XECp}OfGepSW88)u`+c8n(daIWOmzQT}#Sw<O9i#(esr9C}()kd`G-EPPUGE97ixB&LL7snP@@Yb9j?>%tcB79a5qkLezAyKeLruLG2^rEzZMZD$(ABG5{9cG}A13A%Hntvt7q9%R=;JiCel{(n2hA%EcDqt&CyMfTF__M=?muwf#xGS!zZzl!2xhRI<p&opl1}yeN6CaXu2kC>%}@#}sU*yTNW^3rVbEV6jtY<^g@!L>|#mn_hD8t%TECL}U>%C_a`gFq-lv5OFvlfnXP$77Nroh2}7Zvj9-KHb-xH1BX~4_+j5!y}`9rh-lkZ?j}W+K4@!LYT5jOTau_Y86M+vy|NZnEUP)cNKx`_t4My-XJSd#=4>5$+SWK^cvbbI81*EGO$qZ164mIXEiCZ|WjT;SPpWp=t{+#N4`c!913xv?t0eU3cxDOoG(ioh_q*%<O1_}H$l2n_EWStRq%M$)n0(|OT3@C2E5WMXV0RseOL<CBk~Hf<b-v{+7t|45|1@2XXC%q=HnmPxlTMxuWqD5v>n%O007{ulLk@~C<Jq>~wrQir+$Hm&;|~Xt{%J7AQ4px#VNIpY@L6D#I#yDmx`>Mnmqj=+J{+^+ytZRR*ca6JaMYCuIMopE(S-A}Du!pNj$PWg`>Gm_sHV!R$dUmK$WmjMsssv4i1sSSn`-S+t&9~IvYRwTA=Cv<)m0+1Ik!?G*p7Iy^4*XtO~Jvo_@GU}9j~$tvDQmP<<qz<MFcSuS_RgTbNI#%9ljxI`?nY8FLb*;IRgyo4_-Vo9*-sC&rloU&B2UK{WGP|A%y^o^aKW#kX!{;8?xs>m!;#~aN41eW7Xa;Ap6vO5~?unvW1$OWFe=4<WBaS+|8b&?C2(3SeJ!fwFftczUet)Gzr=|RIpqyq6`NKon^|V>4Sbr0J#Uu^}1SKe~lFl-h|wuJ<9lZ39>svB~`Q}YgpnfQqhi}gia7i0I{661=0y-fmty$3D%gQ1`f_y^keE`PTrafs68cgvQ51dvi;4^esK4fpS^SU*LOeu{Qb{A`s~l|{`qG=yZe*7KmYu_&p+(zoJ9?wcVic{!#nJP5KzE6(yU@Jrkw*PwbuH%E@RBpu(~Z6gayRO2|;9P7GDneMUJ+<d(dF=Zmc$w6gLAUi<z1>p#rbn?s_8#CG+HqoT9-k7^UFlib>46)Y|EqwKGwRr+dLHo=GijQ-7L!qop%Kw5YTy><9#i%?vU+Dd)+^0m#`7EwFTp%_6KqqSItPOLrm_O4^a2X?)aLO_<N%4@tI^QV*(VQv-y--v8qFKfeE)pWgrE5BJ~s$^GAb@W+37k66w9AO8N2cmDc|pZwKVAO85upZ&#`KmR9w`_nsL{`~Lm|IM$y{PYj^fB5!SAAfTHzx?W}4}VGDzIgXPfAQ|$ef9Q-_y6+y^!(-jdgsgE{NnzH|MJz3eqFQS`+xl8qx*0F_5EM`>i+NEW!V4q?fXCa{g?mz?pGgweE%<gc>i~w(A&>`_Sp|Ggk*B>)8Ed$&)&Ir=Lg=_-G59F$FRUF)|y_O-2E+H<agi4glygW^n=u!;hR0vG?_AJJ!3q9<tv2|W+~?z=WZ*7YA54KsUauL2QPbp<2voaAg^{QNhbC<b!nPeScOlx^`#Iber8>D)NHF;$RgDU3W}RSLMEg^I?O6m=q!(Vp`saZQ-zwonq!i4^3Z}J%LAtHl4PjkdDky?PvARB#m#jq`+gO?NN-E(Q5Px~Vs17dn=-_T1|-_5l9%Hlg!FV=T&GZcSaXquYzp;9b~MY=HCB4_uR?xae(d%p8jHtAm-)rBN0)K%dRDP}r7>w7cgl)?;xa_ri#docG*iG9WD1a_evm@P|8DD>8}KMQg6Gx|2|I{H6lbiXHx`tW3E6Fl*B()ogmqME+@qsEz5n0;^tZqK<Ntj7{%_v}Z&1%){LQbw`0@X~|D%tN3i4?1b|!Z}CZ=}p&R=2$-uv|Z_(;FH`;W*8iWUC%2OgJg`!(oux~xPdEeR-Gu*6Aw73uQCJH^62ti+U(<se<K;bXEu0P7LlYll0;B=H)ljwnF@rXxz&fbvGT8w!hNU@%7SKrlc0)31N})ra5z^1bgL4Z}ISP(pV`nx{mDfd5AwB=d!vhhOHfk=W5v(A3+b2z$z-OG}jY_NlWW)<7rD#`W!JCDT+Ey&;jc&Z*$AD$8ydZ;py!y&<qE%>@>tQT--W9fg({L{!)?5=7})N|dFz;fB4llQqp|l^dGJR-15`q78}osZ!sRX(W_u8E=TyJeS9%xGwHf&?~}Q67k*QjGeLetIU0=@+MGv94h4Vk-C~G#JJRKa5PeBRrQH5NaRqOn5{@$+v_8%Tos~nsZreu#38Coxlj}KO+oRT8=&W89&iiT>0c>~o+W-=6Ke84Ip-Nj<c+uemd#O}T%;-}UxDpNngo+1LChsH8f>rIZdVU-CSu-XL+I&tI|gP?|KR*HX&&!0JHSisK~Je-4r*NToLmuAU!ppP>19&@*RkDkYJu5pe|?!3R%Ki`Ri=f<y_?ztgFR}16A%)+%bcBn+q1cW<O7dcr!t<g3ly6JiSgh}G)+x@$5Z5O&||U>$_g8X6DbvUiRfjYEO&s)*Q9iiReqQ~?IG260I%@7t%VDVkIp1DaQr-b2tlFA;x-W9Pw}LK!7;Kw;>EGAoZH(ruh5_|r&<NvQ)~k6ZgWbEpxh&S8F?zSY6A3_OHlH=##~0L^Tt$Ig9&@5cNO+`%EYuf7lfNZrZ!Wp*}P6j=0mzAU3IgP38=q1`I_V|WB!Dl2^-<aO5#e=OL4%ee~zV0DOrWeJx;7yj$}0{hmg0Qeh)bH6S*epbVlMGx~vz}9+<f<sGlio9tp-_8His~dk&FVnM@Do8#O8K1J!@Bq#7e?tK4kJ(jckY#o5HQK`6s*`}o*N`>p0e_C85pB@<Vs?j>fcGYqj{RLOTuGpF&qa-wuDcWFk?)g^CX5)Z?v&l66PW*p=`m#kC~rJe;JBCixPw@cZ+btg2PL%1VllVv6%aY2wrzZieD#JDOyx1*#|$|$w1TRJ)`_Y?KQF{?rZ<skeL{in_Q#8yyUy(#Te9_~hNZl|(NjmH|5ZpiSYzCCP$r>o^i%{p9_=ZTxEP4h(Cx0)%{r!wJBdsX_Ym7Sv8NlM@3bf-q&t|r)|&Pn$)kvW-IFCb4mk_C;&ND`!~wB7W&&>G<kzGsI%8ro66HaF~&nw~`LBg?b=YH_j`iy^r@M;|nlf7?=~SN3UHz8i#+2n@*SnA|8Cpn4Fu7L>G4JL;wpP(5cmMbh*cIW=iHnv>R71ZjI=sPR@D^EvZqM**i|5mLED9G+m`ThdakgFP=f$`kyaf)I#3cqw~|AZC+ES_>x)*_I_~>NmV|7fEN)HTS&G6MMF(nXbzMyPV+2j9@@oW|+2iwfp9V9=lIbuA{}&WI@s`n6rfTtf4V1B|TE=acmzgeW^@vgA%UKDM>!1U@+Uca=iMp>SZq+wYx|-V76r)fZ!eZ%4?i{2d!tj2(8cQxQOkS-_eel^NNirZv#{il$QX;Y$%A>=LM-tLUv|W`K};UW=J^6<1bw=?K24z4OpZ^w{TdmZ;>$XPI7TZ_@ufy1N+LzQ8AM>i{HUZZ;DhxJdIWF3@N~y3S7yS0_)XxMzl>#>OFEC?zFn@p#wL;1f)D9zX{fM?ZwIYb5)a)zNjYO9cIO8(s4*h#snMNRmaG&JtetPeShi2!Hwr$y80rmk4jEq5sOs>L~{UyFwdLd8<0sA`@Et~4MW+>J$BY-a|y`h5{^5}t5x<}PefaG$(Yf`T@$iU!)z5s#zQ#jp9UdZ5HmxxD%j|#cQ3ti^~J%pe-GbA>-olW|C;f!Jj4UK>A>QVwNd>|WcWa&_+Zra`i*+(st?XQVUk|I;asSKEJSd&TRM?Qet6B73>UMdtFlfBuVjY*FU~e9>uz27y1cyYN|tO}KHayj6Zv{1)Bj%wFD-mq;;_jvRn3JuZxVz#3sj>dJZ}>GGG{90H3>yadADqWQ<_@Bu6%irbf&BuJWW2+s*Jm_DqCnZ)u>joxiYYmd{mN)K3rbW@tLb>mM!Ki9^)vgaCLlUc^u-pz(*W0+(?Hr?bEQNY*pQ|wX>(g?Jd$htxj{wcS30KTz%8Zbt1&fz46GJhl>8Y?K*(;wJJ&JteRliDlSLIr2w!;>xv5*3`A4Y6<{*k{jh>GfxTHpt?`6$Q?X;T5X*RV9#%Bv6HfU)RJl4;E_llJE&fV;#%6B~Y0(a+(ihmknz`OT#gHg@x=9zV+w7Ma=v50e=<2vSU@PLz;UGnfCYs|o+0+4l!QQ4Op}Of(UGjBF-W_NfFqQO+x;mgDoCRTkFO#-E2+cj_&|F;u23zGI)J&?<TL+xTyTWzFHjCUz>3lI2jV|Z;V<fAMY17Ha-q+LW_*&)7gm04I_j=yal^F@=SXRx{q-*al-?E_wQU<M6?wKHUmKhDjKbQwM%Xq>W+0|)Founx3SF=l>wQMDDp?O(bm{<+6bR`prr(rs$25y}OnW0!$6PSM6uuRP2UKDRngq=~Z_BU;m_7RhfHzhrS{_t#2+b^XQO?03Hrv#nZ%(QcQ_P+KWOSX$iUM_TW)v_n{Yq-XeE@MTO;HCMd+wD4~;ms+p9LJ>0vdan^b<XYYtC`dGqg83fE7$7P0JSNM8ZSrLacG6U8Gcg(N7;r?CMs3%<enM(@M&PGq9;Q&sruCA45uJdf>Z~pYL0e5jVr8)p2qy-jzcQT7E;%UW4&JPumE#S+@#-EE_CfW@~tNWd>fQIplo+~-W6Z3si#ST^K}d=%fQeJ7GM1p3mS}G{sRXdz!<)uwsf8ht?1IrhQs`!NLkGj+GV6pO%HfSXRjW_kc{FmQH}?4n{FWQT8dj}34=MuDWT4vxhQC%N}F^Ka4?MGX)Ac<BG?=Ig(~302iE(iZ31T{x--M9Kh0V$m3l5ED3G;+6k%k$h;t6c<)~jl52~tP$4zEEO|x=-)rr>?A2jZRHz|4iXfce*hef)DjP(&>xjeq;V_%MGNJ`Gm+uj^1EKY_%>eEZGyHMr@N~*ve5Jwqx$#tNWi44@7(i{{hX~c0-QD0oEQ3EwOrV-phSxwebcjFoK>`j-nt(0u4f-}<dj5v+&!%8>WC}ZH%9cR=%T^L<V8Bk2w+Gu^kfRTKV$3QsFjxd%TG5M8OW|FgsJ_VMJCYI}tX4}W@`h?HYU{wK!kJ@kzS(tt3$vdDY>7Mi+3q3<V=u$4ubNUN3z6mQJ+z-tX46SqMg$K~_Z$-3fmBDYMAPOZ#nR*0q!O6q74}T&gOSz63Pu1Vtf82ZV-lu;k;F1r2QBe}9JR0`(J(L*i%$j#~a_`RDxE0F1VhX4FOISd6-(z>Szc18xCh6l#-tg`}6E^SN`CEL@)&(PWK!I_liUn$l(~m*Ku<IDHqYtn}cK5e%bKPH9O_n<g6}WfjpV(3M7YUhEQEV>mvw8q#Q)uYI)q9`*L*ng&9H!)@XZJ+$IO<1&24dVgibl5X-T6<@uPhf<GC|W=Z{54|KhVaV|LArT422XXaCiTJ@yD1qP#gBU{zh=_8x8H;nB4tuGeX=a>{Pw4(WZA*jAr-Vr@z4=HokY~Lzzr(NOXZa#<uU>d1tr`*JrRFB&AAWdiT9DYXqi$^7%(0<z_x$*Tz41I{^Lz>IS{Ncjq5KAaH&Rycfj@a*jeD#2B>{WBUt^JbvAPUIpyW-{(qq-=CLAs%Y(m&X^(;p5Ye#n}zno1dU+EAkN?m1;T-DhHyMXUkPLiYdev)I{KdB)lfykk@xSSEl#g<M6>x<fJ#$$<h{%d9eLj_mj@NGX&28|H13XQu^r)-wl$CcZmj;tw~n*6fMLfq&`Q^AbA0B=JKQfh_pv9>_6bM7D=A99DNhjQm8HzlugbF#-kh=d4TaKDFE%-3>G8*b1|XhD5wjO7HsgQqd*4&rxn&$QRPPGSed`)>sp{8FkIWGoyb2~HYpn35pMP}k&Iif%>DsMCQq{L>^hnE|fBwIH{mM(%Z~ceC&1<i|a`RR&8#$L|OCV0oUE}rQ4>@s<Nlae8dIp{}Y!a?GUMhXNF2CX?v5F?YM$ThfrgFxqR(CycmbFWb8l*ZkQJrR|R`i4KPUF2fOBs*UoH`kgSj(iv&*RGV@N7J;xjsi3P{dyA*ic}vy~-EmDeoQSpCm>t@j~I7k}|sghQrY1N(z$&xxBl=Tw+}sf%OfSRZC5I>kxX|aBDhUyLhS_(r@a4Y4`SAjYUO_8)qb=Z>}{DsqUZ3`>ibdwb@|=V!py0Q-2SorF!OzQ6w#od-G(y4a;^guUw90#wWLh$bwNfZ`cUb0;Y6`=Cjvu4PxExSk~Kt<HOgh@!&i49gzCjz8BU^mrCnc>u+QsfkMoeYjr#~<*Cudb7oJ{X(=WBMrnjHr;(OrBFwFr$wpc_wUtn(Uy+P65}m8BfVIY;ta7-_0~fKh{3jF@EC=JdmN)Vabc`5kxT4j!uOG8-!>cp7HJaOi)g0VJ5r0q{<9mX+P3{3Sqdn<Lxre&}@ZoR&t8rmd20GPcYPo6+d%$b*{_n2HJN@-|WuEefy0Ni20qNI&!Qp8yIeg%Y4o~Y9Ut<ye*B_GKG|-0#(XSc*Lxt?0)TP=G{)U9It3ni-3F5?Xg^uTTrdJ-OYeFg9?T|apTjJ^~9F1QEd(72b6uH`C&~g?efshVXW&@PCPmizjmbzE6ZI*~t_W4mHTQr2j-hhq017p+KBKsg03vg!_!fdzXDW_UUHbi-XK$3pP1R&ZhXwLSlEv_o{5ax=Jgqex8oAbv|^{D0JZg9DFxuZ?-J-*ucEVjfQI&-n)4hV3oFPeeS#gh$vT5b?oQYdJUlElSHQ8rBRBF(iTPB^+;%V<BB>*$;3oqCgKI`8?{^##syhjuA)`>Ua)^BQ2+^o`FY7!OI(Kn0v{vl%4c|G6f&Eej~Q$5uj)APMCQjz%3Xro1srROgyvjvio>vz<yoO2^2uDdc|`C=})J4o-$WMF}ftg*Aa+&m=2H>H2wYo-qk6QI+`GC97J{OdTfJq^`a5jn&}ay@vCRRci(QY(KmeNEFWTa9CV!RG_VHPWazRD_oJ;>89_2J1Kq8a(u=aS`;JWe+YWsYP@#5B$ss#X{WuqQbdym7wFa$uxLPlR4>-df=EP0)!JE;UVcVV-dnH=rE){sC6&EDNs}bn4(XL|jV+*2+isOH-BT;^Z@l{s2Al$5pm$$#IL(U@ojYllp@1K#8kSEZ`b-$}{igp1Akmqu'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if len(SAMPLES) != 97 or len({r['domain'] for r in SAMPLES}) != 97:
    raise ValueError('Phải có đúng 1 mẫu cho mỗi hostname.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != 'fixed_97_hostnames'
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp 97 mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': 'fixed_97_hostnames', 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': 'embedded samples from data/links_corpus.parquet',
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': 'equal_host_reservoir', 'seed': 42, 'selected': len(SAMPLES),
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: s['domain']):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_'):
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = classify_error(status, detail, page_http)
            reason = STATUS_REASONS.get(status, status)
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
        })
    return rows

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    write_csv('ket_qua.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': n,
               'domains': '; '.join(r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind)}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'access_result',
                      'http_status', 'robots_http_status', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được = HTTP trang cuối 2xx; QC là kết quả riêng. Dừng do robots là chính sách crawler. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoặc khi hết lượt. Cùng domain gốc được giới hạn
tốc độ; robots.txt được kiểm tra trước mỗi origin. Lỗi của một URL không
dừng các URL khác. Báo cáo và ZIP cập nhật khi ô kết thúc, kể cả khi có lỗi.
Nếu dùng `MAX_NEW`, phần chưa chạy vẫn được ghi riêng, không tính là thất bại.


In [ ]:
import argparse

try:
    crawler.locked_crawl(argparse.Namespace(
        out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
    ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
ZIP gồm toàn bộ kết quả, raw và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
